# ApprenticeOps — wave analysis (live snapshot)

Re-runnable analysis over a **committed snapshot** of the benchmark run.

Two views:
1. **Burn-through time per bracket** — stacked columns of wall-clock minutes (stacked by model). Bigger models cost more; this shows roughly how much longer a heavier bracket takes to grind through.
2. **Precision @ 0.7** — stacked columns of pass/fail per bracket, using a deterministic-score threshold of **0.7**.

The snapshot lives at `data/snapshots/results_snapshot.csv` (text-free: model, bracket, scenario, rep, det_score, decode_tok_s, wall_s, dnf, finish_reason). Refresh it from the run node any time (see the last cell).

In [ ]:
# Dependencies (no-op if already installed)
%pip install -q pandas matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PRECISION_THRESHOLD = 0.7  # deterministic-score pass bar
BRACKET_ORDER = ['0-1B', '1-2B', '2-3B', '3-4B', '4-5GB']

# Find the committed snapshot regardless of where the kernel started.
candidates = [
    Path('data/snapshots/results_snapshot.csv'),
    Path('../data/snapshots/results_snapshot.csv'),
    Path('../../data/snapshots/results_snapshot.csv'),
]
snapshot = next((p for p in candidates if p.exists()), None)
if snapshot is None:
    raise FileNotFoundError('results_snapshot.csv not found; run the refresh cell at the bottom.')
print('Loading', snapshot.resolve())

df = pd.read_csv(snapshot)
for c in ['det_score', 'decode_tok_s', 'wall_s', 'rep']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['bracket'] = pd.Categorical(df['bracket'], categories=BRACKET_ORDER, ordered=True)
present = [b for b in BRACKET_ORDER if (df['bracket'] == b).any()]
print(f'rows={len(df)}  models={df["model"].nunique()}  brackets={present}')
df.head()

## 1. Burn-through time per bracket (stacked by model)

Column height = total wall-clock minutes spent in that bracket so far; each segment is one model. The annotation shows the **per-model average**, which is the apples-to-apples cost (brackets can have different model counts).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
avg_per_model = {}
for i, b in enumerate(present):
    sub = df[df['bracket'] == b].groupby('model')['wall_s'].sum().sort_values(ascending=False)
    bottom = 0.0
    for model, val in sub.items():
        ax.bar(i, val / 60, bottom=bottom / 60, width=0.62, edgecolor='white', linewidth=0.5)
        bottom += val
    n_models = max(len(sub), 1)
    avg_per_model[b] = (bottom / 60) / n_models
    ax.text(i, bottom / 60 + max(bottom / 60 * 0.01, 1),
            f'{bottom/60:.0f} min\n{n_models} mdl · {avg_per_model[b]:.0f}/mdl',
            ha='center', va='bottom', fontsize=9)
ax.set_xticks(range(len(present)))
ax.set_xticklabels(present)
ax.set_ylabel('wall-clock minutes (stacked by model)')
ax.set_title('Burn-through time per bracket (stacked by model)')
ax.margins(y=0.12)
plt.tight_layout()
plt.show()

# Quick ratio sense-check (e.g. 1-2B vs 3-4B)
if '1-2B' in avg_per_model and '3-4B' in avg_per_model and avg_per_model['1-2B']:
    r = avg_per_model['3-4B'] / avg_per_model['1-2B']
    print(f"avg/model: 1-2B={avg_per_model['1-2B']:.1f} min, 3-4B={avg_per_model['3-4B']:.1f} min  ->  3-4B is {r:.1f}x slower")

## 2. Precision @ 0.7 per bracket (stacked pass / fail)

Each (model × scenario × rep) result counts as **pass** if its deterministic score ≥ 0.7, else **fail**. Columns are stacked pass+fail; the annotation is the pass-rate.

In [ ]:
passes, fails, rates = [], [], []
for b in present:
    sub = df[(df['bracket'] == b) & df['det_score'].notna()]
    p = int((sub['det_score'] >= PRECISION_THRESHOLD).sum())
    t = int(len(sub))
    passes.append(p)
    fails.append(t - p)
    rates.append(p / t if t else 0.0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(present, passes, width=0.62, label=f'pass (det ≥ {PRECISION_THRESHOLD})', color='#2e7d32')
ax.bar(present, fails, bottom=passes, width=0.62, label=f'fail (< {PRECISION_THRESHOLD})', color='#c62828')
for i, b in enumerate(present):
    total = passes[i] + fails[i]
    ax.text(i, total + max(total * 0.01, 0.5), f'{rates[i]*100:.0f}% pass', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('result count (model × scenario × rep)')
ax.set_title(f'Deterministic precision @ {PRECISION_THRESHOLD} per bracket (stacked pass/fail)')
ax.legend()
ax.margins(y=0.12)
plt.tight_layout()
plt.show()

## 3. Bracket value gate (pre-registered)

Should the expensive **4-5GB** bracket earn a slot in a later wave? It costs ~3× the 1-2B bracket per model, so we gate it on **judged** quality (not deterministic checks).

**Rule (fixed before looking):** RUN 4-5GB **iff** its judged %-of-frontier beats **3-4B** by **≥ 5 points** *and* their **95% CIs don't overlap**; else **HOLD** and report the **3-4B Pareto knee** as a finding. The **guard (safety) class is always run** regardless. This cell stays **PENDING** until a judged snapshot (`data/snapshots/judged_snapshot.csv`) exists.

In [ ]:
# --- Bracket value gate (pre-registered) ---
# Decide whether the costly 4-5GB bracket earns a slot in a later wave.
# RULE (fixed before looking): RUN 4-5GB iff its judged %-of-frontier beats 3-4B
# by >= GATE_MIN_LIFT AND their 95% CIs do not overlap. Else HOLD.
# Uses the JUDGE metric (not deterministic checks); the guard class is always run.
import numpy as np

GATE_MIN_LIFT = 0.05  # 5 percentage points of %-of-frontier (judge_score / 5)
GATE_REF, GATE_CANDIDATE = '3-4B', '4-5GB'

jcands = [Path('data/snapshots/judged_snapshot.csv'),
          Path('../data/snapshots/judged_snapshot.csv'),
          Path('../../data/snapshots/judged_snapshot.csv')]
jpath = next((p for p in jcands if p.exists()), None)

if jpath is None:
    print('Gate PENDING — no judged snapshot yet.')
    print('Expected data/snapshots/judged_snapshot.csv with columns: '
          'model,bracket,scenario,rep,judge_score(1-5).')
else:
    jdf = pd.read_csv(jpath)
    jdf['judge_score'] = pd.to_numeric(jdf['judge_score'], errors='coerce')
    jdf = jdf.dropna(subset=['judge_score'])
    jdf['pof'] = jdf['judge_score'] / 5.0  # %-of-frontier proxy

    def boot_ci(x, n=10000, seed=0):
        x = np.asarray(x, float)
        if x.size == 0:
            return np.nan, np.nan, np.nan
        rng = np.random.default_rng(seed)
        bs = rng.choice(x, size=(n, x.size), replace=True).mean(axis=1)
        return x.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)

    rows = []
    for b in BRACKET_ORDER:
        sub = jdf.loc[jdf['bracket'] == b, 'pof']
        if len(sub):
            m, lo, hi = boot_ci(sub)
            rows.append((b, len(sub), m, lo, hi))
    summ = pd.DataFrame(rows, columns=['bracket', 'n', 'pof_mean', 'ci_lo', 'ci_hi'])
    print(summ.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(summ['bracket'], summ['pof_mean'] * 100,
           yerr=[(summ['pof_mean'] - summ['ci_lo']) * 100,
                 (summ['ci_hi'] - summ['pof_mean']) * 100],
           capsize=5, color='#1565c0')
    ax.set_ylabel('judged % of frontier')
    ax.set_title('Judged quality per bracket (95% CI)')
    ax.margins(y=0.12)
    plt.tight_layout()
    plt.show()

    def g(b, col):
        r = summ.loc[summ['bracket'] == b, col]
        return float(r.iloc[0]) if len(r) else None

    cand_m, cand_lo = g(GATE_CANDIDATE, 'pof_mean'), g(GATE_CANDIDATE, 'ci_lo')
    ref_m, ref_hi = g(GATE_REF, 'pof_mean'), g(GATE_REF, 'ci_hi')
    if None in (cand_m, ref_m):
        print(f'\nGate PENDING: missing judged data for {GATE_REF} or {GATE_CANDIDATE}.')
    else:
        lift = cand_m - ref_m
        non_overlap = cand_lo > ref_hi
        run = (lift >= GATE_MIN_LIFT) and non_overlap
        print(f'\nGATE {GATE_CANDIDATE} vs {GATE_REF}: lift={lift*100:+.1f} pts '
              f'(need >= {GATE_MIN_LIFT*100:.0f}); CI non-overlap={non_overlap}')
        print('VERDICT:', 'RUN 4-5GB in the next wave'
              if run else 'HOLD 4-5GB (no judged lift over 3-4B) — report the 3-4B Pareto knee')


## 4. Roofline / cross-hardware extrapolation (PAPER §7c)

Decode is **memory-bandwidth-bound**, so throughput transfers across CPUs by the **bandwidth ratio**, not clock or a CPU-mark score. This cell fits the node's measured bandwidth efficiency (MBU) and exposes `predict_decode_tok_s(...)` + `extrapolate_tok_s(...)`, reporting **intervals** and the **regime/ISA/KV caveats**.

> **Honesty (PAPER §7c):** one node ⇒ the hardware step is a **physics rule, not a fit**. It is a **hypothesis until validated out-of-sample** on ≥1 real CPU (run `calibrate.py` there + spot-check 2–3 models). Needs the richer snapshot columns (`size_bytes`, `param_count`, `membw_peak_mb_s`).

In [ ]:
# --- Roofline / cross-hardware extrapolation (PAPER §7c) ---
# Decode is memory-bandwidth-bound: tok/s ~= MBU * B / W, with W ~= bytes read per
# token (weights at fixed context). We FIT only the model-axis (MBU efficiency on
# THIS node); the cross-hardware step is a first-order physics rule, NOT a fit.
# Findings actioned: report INTERVALS not point estimates; regime guard for sub-1B;
# fixed-context (KV) caveat; out-of-sample validation required.
import numpy as np

NODE_PEAK_GBS = 38.4  # i5-8350U dual-channel DDR4-2400 theoretical peak (this node)

need = {'decode_tok_s', 'membw_peak_mb_s', 'size_bytes', 'param_count'}
if not need.issubset(df.columns):
    print('Roofline PENDING — snapshot lacks roofline columns; re-run the refresh cell '
          '(the richer export adds size_bytes/param_count/membw_peak_mb_s/energy_wh).')
else:
    rf = df.dropna(subset=['decode_tok_s', 'membw_peak_mb_s', 'size_bytes']).copy()
    rf = rf[(rf['decode_tok_s'] > 0) & (rf['size_bytes'] > 0)]
    # Measured bandwidth efficiency (transfers within an ISA + memory-topology class).
    rf['mbu'] = (rf['membw_peak_mb_s'] / 1000.0) / NODE_PEAK_GBS
    mbu_med = float(rf['mbu'].median())
    mbu_lo, mbu_hi = (float(rf['mbu'].quantile(q)) for q in (0.10, 0.90))
    print(f'Measured decode MBU on this node: median={mbu_med:.2f} '
          f'(10-90%: {mbu_lo:.2f}-{mbu_hi:.2f}), peak ref={NODE_PEAK_GBS} GB/s')

    def predict_decode_tok_s(target_bw_gbs, size_bytes, mbu=mbu_med):
        """First-order roofline prediction (fixed context, same ISA class)."""
        active_b = size_bytes / 1e9
        if active_b < 0.7:  # regime guard: sub-1B hits a fixed per-request floor
            print(f'  [regime warning] ~{active_b:.2f} GB model: decode may be '
                  f'overhead/prefill-bound, not bandwidth-bound — prediction is a ceiling.')
        return target_bw_gbs * mbu / active_b  # GB/s * eff / GB-per-token = tok/s

    def extrapolate_tok_s(tok_s_old, bw_old_gbs, bw_new_gbs):
        """Scale an OBSERVED tok/s to a new CPU by the bandwidth ratio (not clock)."""
        return tok_s_old * (bw_new_gbs / bw_old_gbs)

    # Worked example: extrapolate this node's models to a Pi 5 (~17 GB/s) and a
    # DDR5 box (~70 GB/s). Report an INTERVAL from the MBU spread, not a point.
    ex = (rf.groupby('model')
            .agg(bracket=('bracket', 'first'),
                 obs_tok_s=('decode_tok_s', 'mean'),
                 size_gb=('size_bytes', lambda s: s.mean() / 1e9))
            .reset_index().sort_values('size_gb'))
    targets = {'Pi5 ~17GB/s': 17.0, 'this node ~38GB/s': NODE_PEAK_GBS, 'DDR5 ~70GB/s': 70.0}
    print('\nBandwidth-ratio extrapolation (observed -> target), tok/s:')
    for _, r in ex.iterrows():
        preds = '  '.join(
            f'{name}: {extrapolate_tok_s(r.obs_tok_s, NODE_PEAK_GBS, bw):.1f}'
            for name, bw in targets.items())
        print(f'  {r.model:<32} ({r.size_gb:.1f} GB)  obs={r.obs_tok_s:5.1f}  ->  {preds}')

    print('\nCAVEATS (PAPER §7c): single node => physics rule, NOT a fit — VALIDATE '
          'out-of-sample on >=1 real CPU (run calibrate.py + spot-check 2-3 models). '
          'Holds in the decode-bandwidth-bound regime, within an ISA+memory class '
          '(AVX2<->AVX2); AVX-512/ARM/NUMA widen error; extrapolate at FIXED context '
          '(KV traffic grows with c). Clock / CPU-mark is NOT a valid predictor.')


## Refresh the snapshot

Re-pull the current run from the node into the committed CSV, then re-run the cells above. Run this from the repo root in a terminal:

```bash
ssh dragos@home-ai.hont.ro 'python3 - <<"PY"
import json,sys,csv
out=csv.writer(sys.stdout)
out.writerow(["model","bracket","scenario","rep","det_score","decode_tok_s","wall_s","dnf","finish_reason"])
for line in open("/tmp/sme-var/results.var.jsonl"):
    line=line.strip()
    if not line: continue
    try: d=json.loads(line)
    except: continue
    fr=d.get("gen_ai.response.finish_reasons")
    if isinstance(fr,list): fr=",".join(map(str,fr))
    out.writerow([d.get("model"),d.get("bracket"),d.get("scenario"),d.get("rep"),d.get("det_score"),d.get("decode_tok_s"),d.get("wall_s"),d.get("dnf"),fr])
PY' > data/snapshots/results_snapshot.csv
```